# Import packages and load data

In [ ]:
import pandas as pd
import numpy as np
import time
import sklearn
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib import rc
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator) 
from matplotlib.colors import ListedColormap, Normalize
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, WhiteKernel, ConstantKernel as C, DotProduct)
from sklearn.cluster import KMeans
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", message="X has feature names, but GaussianProcessRegressor was fitted without feature names")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
file_path = '../../chemical-featurization/likeness_data/repeat_rdkit_physchem.xlsx'
model_output = 'gisaxs_fwhm'
# import data frame for pre-processing
df = pd.read_excel(file_path)

In [ ]:
# Drop rows with any NaN values
df_clean = df.dropna()

# Assign input variables and target variable
inputs = df_clean.iloc[:, 1:-2].copy()
target = df_clean[f'{model_output}']

# GPR training

In [ ]:
initial_seeds = [1221, 1410, 1919, 24, 81, 21]
total_seeds = 10

rng = np.random.default_rng(42)  # reproducible randomness

remaining = total_seeds - len(initial_seeds)
if remaining < 0:
    raise ValueError("You provided more seeds than the total requested.")

# Generate random seeds that do NOT overlap with initial_seeds
random_seeds = set()
while len(random_seeds) < remaining:
    candidate = int(rng.integers(0, 10_000))
    if candidate not in initial_seeds:
        random_seeds.add(candidate)

all_seeds = initial_seeds + list(random_seeds)

assert len(all_seeds) == total_seeds

In [ ]:
%%time

# Train/test split
X_data, X_test, y_data, y_test = train_test_split(
    inputs, target, test_size=0.2, random_state=10
)

# Normalize inputs globally
scaler = StandardScaler()
X_data_scaled = pd.DataFrame(
    scaler.fit_transform(X_data), 
    columns=X_data.columns, 
    index=X_data.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), 
    columns=X_test.columns, 
    index=X_test.index
)

n_samples = len(y_data)

# Seeds for different initializations
initial_seeds = all_seeds
all_r2_scores = {}
n_initial = 10
n_iterations = 10
batch_size = 10
top_m_factor = 10

# Define Gaussian Process kernel
kernel = (

    C(1.0, (1e-3, 1e3)) * DotProduct(sigma_0=1.0)
    + C(1.0, (1e-2, 1e2)) * RBF(
        length_scale=np.ones(n_features),
        length_scale_bounds=(1e-2, 50.0)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e-1)
    )
)

for seed in initial_seeds:
    np.random.seed(seed)
    initial_indices = np.random.choice(n_samples, n_initial, replace=False)

    # Initialize training set and pool
    X_train = X_data_scaled.iloc[initial_indices].to_numpy()
    y_train = y_data.iloc[initial_indices].to_numpy()

    mask = np.ones(n_samples, dtype=bool)
    mask[initial_indices] = False
    X_pool = X_data_scaled.iloc[mask]
    y_pool = y_data.iloc[mask]

    
    # Reinitialize GPR for this seed
    gpr = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        alpha=1e-10,
        random_state=42
    )

    r2_scores_seed = []
    selected_indices_seed = [initial_indices.copy()]

    for i in range(n_iterations):
        gpr.fit(X_train, y_train)
    
        y_pred, y_std = gpr.predict(X_test_scaled, return_std=True)
        r2 = r2_score(y_test, y_pred)
        r2_scores_seed.append(r2)
    
        # Print seed and iteration R2
        print(f"Seed {seed}, Iteration {i+1:03d}: n_train={len(X_train)}, R2={r2:.3f}")
    
        # Batch acquisition: uncertainty + k-means diversity
        _, sigma = gpr.predict(X_pool, return_std=True)
        
        top_m = min(len(sigma), batch_size * top_m_factor)
        top_indices = np.argsort(sigma)[-top_m:]
        
        X_candidates = X_pool.iloc[top_indices]
        sigma_candidates = sigma[top_indices]
        
        # K-means clustering in feature space
        kmeans = KMeans(n_clusters=batch_size, random_state=seed)
        labels = kmeans.fit_predict(X_candidates)
        
        selected_batch_indices = []
        
        for k in range(batch_size):
            cluster_mask = labels == k
            if not np.any(cluster_mask):
                continue
        
            cluster_indices = np.where(cluster_mask)[0]
            best_local = cluster_indices[np.argmax(sigma_candidates[cluster_indices])]
            selected_batch_indices.append(top_indices[best_local])
        
        # Update training set with batch
        X_next = X_pool.iloc[selected_batch_indices].to_numpy()
        y_next = y_pool.iloc[selected_batch_indices].to_numpy()
        
        X_train = np.vstack([X_train, X_next])
        y_train = np.append(y_train, y_next)
        
        # Remove selected points from pool
        X_pool = X_pool.drop(X_pool.index[selected_batch_indices])
        y_pool = y_pool.drop(y_pool.index[selected_batch_indices])
        
        selected_indices_seed.append(selected_batch_indices)

    
    # Store R2 scores after finishing all iterations for this seed
    all_r2_scores[seed] = r2_scores_seed

In [ ]:
import pandas as pd

# Define metadata explicitly
experiment_name = "fwhm_gpr_batch10"
initial_train_size = 10
batch_size = 10

# --- Build export DataFrame ---
df_export = pd.DataFrame({
    "n_train": x,
    "r2_mean": r2_mean,
    "r2_std": r2_std
})

# Save
csv_name = f"{experiment_name}_learning_curve.csv"
df_export.to_csv(csv_name, index=False)

print(f"Saved: {csv_name}")

In [ ]:
# Convert all_r2_scores dict to a 2D array
seeds = sorted(all_r2_scores.keys())
r2_matrix = np.array([all_r2_scores[seed] for seed in seeds])

# Compute mean and std across seeds
r2_mean = r2_matrix.mean(axis=0)
r2_std = r2_matrix.std(axis=0)

initial_train_size = 10
batch_size = 5
n_iterations = r2_matrix.shape[1]

x = initial_train_size + batch_size * np.arange(n_iterations)

# Fine x-grid for smooth ribbon
x_fine = np.linspace(x[0], x[-1], 1000)
r2_mean_fine = np.interp(x_fine, x, r2_mean)
r2_upper_fine = np.interp(x_fine, x, r2_mean + r2_std)
r2_lower_fine = np.interp(x_fine, x, r2_mean - r2_std)

# Colormap
norm = colors.Normalize(vmin=np.min(x_fine), vmax=np.max(x_fine))
cmap = cm.viridis
colors_fine = cmap(norm(x_fine))

fig, ax = plt.subplots(figsize=(8, 8))
%matplotlib inline
rc('text', usetex=False)
rc('mathtext', fontset='cm')
rc('xtick', labelsize=25)   
rc('xtick.major', size=7)  
rc('xtick.minor', size=4)
rc('xtick.major', width=2)
rc('xtick.minor', width=2)
rc('ytick.major', width=2)
rc('ytick.minor', width=2)
rc('ytick', labelsize=25)
rc('ytick.major', size=7)
rc('ytick.minor', size=4)
rc('axes', labelsize=25) 
rc('axes', linewidth=2) 
rc('font',family='sans serif')
rc('font', style='normal')
rc('font', weight='500')
rc('font', size='15')
rc('axes', labelweight='500')
rc('axes.spines', **{'right':True, 'top':True}) 
plt.rcParams['font.family'] = 'Arial'

# Scatter points for mean R2
sc = ax.scatter(x, r2_mean, c=x, cmap=cmap, norm=norm, s=100, edgecolor='grey', alpha=0.9, zorder=2)

# Draw smooth shaded ribbon
for i in range(len(x_fine)-1):
    ax.fill_between(
        x_fine[i:i+2],
        r2_lower_fine[i:i+2],
        r2_upper_fine[i:i+2],
        color=colors_fine[i],
        alpha=0.3,
        linewidth=0,
        zorder=1
    )

# Axes formatting
ax.xaxis.set_major_locator(MultipleLocator(25))
ax.xaxis.set_minor_locator(AutoMinorLocator(5))
ax.yaxis.set_minor_locator(AutoMinorLocator(2))

# Colorbar
colorbar = plt.colorbar(sc)
colorbar.set_label("Training set size", fontsize=20)
colorbar.ax.yaxis.set_tick_params(color='black', width=1.5)
colorbar.outline.set_edgecolor('black')

# Labels and layout
plt.xlabel("Number of training points", fontsize=25)
plt.ylabel("Testing R²", fontsize=25)
plt.ylim(-0.2, 1.05)
plt.grid(False)
plt.tight_layout()
# plt.savefig('gpr_training_curve_5batch_10seed_error.pdf', format='pdf')
plt.show()